In [1]:
%pip install pandas numpy beautifulsoup4 scikit-learn spacy transformers torch pyspellchecker tqdm
# This command installs several Python libraries that are commonly used for data manipulation, natural language processing, machine learning, and deep learning.


Note: you may need to restart the kernel to use updated packages.


In [2]:
!python -m spacy download en_core_web_sm
#download the small English model for spaCy, which is a popular natural language processing library. This model includes pre-trained word vectors and various linguistic annotations that can be used for tasks such as tokenization, part-of-speech tagging, named entity recognition, and more.

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 2.1 MB/s eta 0:00:06
     --- ------------------------------------ 1.0/12.8 MB 2.6 MB/s eta 0:00:05
     --- ------------------------------------ 1.0/12.8 MB 2.6 MB/s eta 0:00:05
     ---- ----------------------------------- 1.6/12.8 MB 1.7 MB/s eta 0:00:07
     ------- -------------------------------- 2.4/12.8 MB 2.2 MB/s eta 0:00:05
     --------- ------------------------------ 3.1/12.8 MB 2.4 MB/s eta 0:00:05
     ----------- ---------------------------- 3.7/12.8 MB 2.4 MB/s eta 0:00:04
     ------------- -------------------------- 4.5/12.8 MB 2.5 MB/s eta 0:00:04
     -------------- ------------------------- 4.7/12.8 MB 2.4 MB/s eta 0:00:04
     --------------- ------------------------ 5.0/12.8 MB 2.5 MB/s eta 0:

In [3]:
#loading the dataset and preprocessing it for sentiment analysis. The code reads a CSV file containing reviews, selects relevant columns, and creates a new column for sentiment based on the review scores. It then filters out neutral reviews and takes a random sample of the data for further analysis.

import pandas as pd
import numpy as np

df = pd.read_csv("Reviews.csv")

df = df[["Score", "Text"]].dropna()
df["sentiment"] = df["Score"].apply(
    lambda x: "positive" if x >= 4 else "negative" if x <= 2 else "neutral"
)

# We will focus on binary sentiment classification, so we will filter out neutral reviews.
# This will help us create a clearer distinction between positive and negative sentiments for our analysis and modeling.
#use only a subset of the data for faster processing and to avoid memory issues. 

sample_size = min(20000, len(df))
data = df.sample(sample_size, random_state=42).reset_index(drop=True)

data.head()


,Score,Text,sentiment
0,5,Having tried a couple of other brands of glute...,positive
1,5,My cat loves these treats. If ever I can't fin...,positive
2,3,A little less than I expected. It tends to ha...,neutral
3,2,"First there was Frosted Mini-Wheats, in origin...",negative
4,5,and I want to congratulate the graphic artist ...,positive


In [4]:
#inspect noise - performing some basic data quality checks on the review texts to identify potential issues such as HTML tags, repeated characters, and spelling errors. This will help us understand the nature of the data and inform our preprocessing steps for sentiment analysis. 

import re
from bs4 import BeautifulSoup
from spellchecker import SpellChecker
from collections import Counter

def has_html(text):
    return bool(re.search(r"<[^>]+>", str(text)))

def has_repeated_chars(text):
    return bool(re.search(r"(.)\1{2,}", str(text)))

data["has_html"] = data["Text"].apply(has_html)
data["has_repeated_chars"] = data["Text"].apply(has_repeated_chars)

print("Total reviews:", len(data))
print("Reviews with HTML:", data["has_html"].sum())
print("Reviews with repeated characters:", data["has_repeated_chars"].sum())

# Spelling error inspection on a smaller sample
spell = SpellChecker()
words = re.findall(r"[a-zA-Z]+", " ".join(data["Text"].head(1000)).lower())
unknown_words = spell.unknown(words)

print("Possible spelling errors:", list(unknown_words)[:30])


Total reviews: 20000
Reviews with HTML: 5147
Reviews with repeated characters: 3629
Possible spelling errors: ['popcornopolis', 'yr', 'palletable', 'caf', 'pantsy', 'coffeed', 'quacamole', 'dont', 'oryzae', 'approx', 'greenies', 'wasn', 'everday', 'www', 'garten', 'orginal', 'morningstar', 'ypoef', 'esp', 'cheez', 'krispies', 'takingit', 'lavazza', 'weightwatcher', 'twinings', 'mins', 'poper', 'tsq', 'oreos', 'cusine']


In [5]:
#Rule-based cleaning - Based on the inspection, we can see that there are some reviews containing HTML tags and repeated characters, which can introduce noise into our sentiment analysis. Additionally, there are some potential spelling errors in the reviews. To address these issues, we will implement a rule-based cleaning function that removes HTML tags, normalizes repeated characters, and retains important punctuation that can affect sentiment (like exclamation marks and question marks). This will help us create a cleaner dataset for our sentiment analysis tasks.

sentiment_tokens = {
    "not", "no", "never", "none", "nothing", "neither", "nor",
    "cannot", "can't", "won't", "don't", "didn't", "isn't", "wasn't",
    "good", "great", "excellent", "bad", "awful", "terrible",
    "love", "loved", "hate", "hated", "best", "worst"
}

def rule_based_clean(text):
    text = str(text).lower()
    
    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # Normalize repeated characters: soooo -> soo
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    
    # Keep letters, apostrophes, ! and ? because they can affect sentiment
    text = re.sub(r"[^a-zA-Z!?']", " ", text)
    
    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

data["rule_clean"] = data["Text"].apply(rule_based_clean)

data[["Text", "rule_clean"]].head()


,Text,rule_clean
0,Having tried a couple of other brands of glute...,having tried a couple of other brands of glute...
1,My cat loves these treats. If ever I can't fin...,my cat loves these treats if ever i can't find...
2,A little less than I expected. It tends to ha...,a little less than i expected it tends to have...
3,"First there was Frosted Mini-Wheats, in origin...",first there was frosted mini wheats in origina...
4,and I want to congratulate the graphic artist ...,and i want to congratulate the graphic artist ...


In [6]:
#statistical cleaning - After applying rule-based cleaning, we can further refine our dataset by using statistical methods to identify and retain only the most relevant words for sentiment analysis. This involves using techniques like TF-IDF to filter out extremely rare words that may not contribute much to the sentiment classification and extremely common words that may not carry significant meaning. By doing this, we can create a more focused dataset that emphasizes the most informative features for our sentiment analysis tasks.

from sklearn.feature_extraction.text import TfidfVectorizer

# Remove extremely rare words and extremely common words
tfidf = TfidfVectorizer(
    min_df=5,
    max_df=0.60,
    max_features=30000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b"
)

tfidf_matrix = tfidf.fit_transform(data["rule_clean"])
tfidf_vocab = set(tfidf.get_feature_names_out())

def statistical_clean(text):
    tokens = text.split()
    kept = [
        token for token in tokens
        if token in tfidf_vocab or token in sentiment_tokens
    ]
    return " ".join(kept)

data["stat_clean"] = data["rule_clean"].apply(statistical_clean)

data[["rule_clean", "stat_clean"]].head()


,rule_clean,stat_clean
0,having tried a couple of other brands of glute...,having tried couple of other brands of gluten ...
1,my cat loves these treats if ever i can't find...,my cat loves these treats if ever can't find h...
2,a little less than i expected it tends to have...,little less than expected tends have muddy tas...
3,first there was frosted mini wheats in origina...,first there was frosted mini wheats in origina...
4,and i want to congratulate the graphic artist ...,want graphic for putting entire product name o...


In [7]:
#spacy preprocessing - We will use spaCy, a powerful natural language processing library, to further preprocess our review texts. This will involve tokenization, lemmatization, and part-of-speech tagging. We will also ensure that we retain important sentiment-bearing words while filtering out stop words and punctuation. This step will help us create a more linguistically informed dataset for our sentiment analysis tasks.


import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def spacy_preprocess(text):
    doc = nlp(str(text))
    cleaned_tokens = []
    
    for token in doc:
        word = token.text.lower()
        lemma = token.lemma_.lower()
        
        if word in sentiment_tokens:
            cleaned_tokens.append(word)
        elif token.is_stop:
            continue
        elif token.is_punct or token.is_space:
            continue
        elif token.is_alpha:
            cleaned_tokens.append(lemma)
    
    return " ".join(cleaned_tokens)

data["spacy_clean"] = data["rule_clean"].apply(spacy_preprocess)

data[["rule_clean", "spacy_clean"]].head()


,rule_clean,spacy_clean
0,having tried a couple of other brands of glute...,have try couple brand gluten free sandwich coo...
1,my cat loves these treats if ever i can't find...,cat love treat find house pop bolt hide come t...
2,a little less than i expected it tends to have...,little expect tend muddy taste not expect say ...
3,first there was frosted mini wheats in origina...,frosted mini wheat original size frosted mini ...
4,and i want to congratulate the graphic artist ...,want congratulate graphic artist put entire pr...


In [8]:
#transformer-based confidence - Finally, we will use a pre-trained transformer model to evaluate the sentiment of the original and cleaned review texts. This will allow us to compare the confidence scores of the sentiment predictions across different preprocessing methods and understand how each step affects the model's ability to correctly classify the sentiment of the reviews. We will use a model fine-tuned for sentiment analysis, such as DistilBERT, to get these confidence scores.

from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Use a smaller subset because transformers are slower
eval_data = data.sample(min(300, len(data)), random_state=42).copy()

def transformer_confidence(text):
    result = sentiment_model(str(text)[:512])[0]
    return result["label"], result["score"]

for col in ["Text", "rule_clean", "stat_clean", "spacy_clean"]:
    labels = []
    scores = []
    
    for text in eval_data[col]:
        label, score = transformer_confidence(text)
        labels.append(label)
        scores.append(score)
    
    eval_data[col + "_pred"] = labels
    eval_data[col + "_confidence"] = scores

eval_data[
    [
        "Text_confidence",
        "rule_clean_confidence",
        "stat_clean_confidence",
        "spacy_clean_confidence"
    ]
].mean()


c:\Users\USER\Desktop\ACADEMICS\nlp\Product and Text review processing\Nlp_assignment1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5170.30it/s]


Text_confidence           0.970699
rule_clean_confidence     0.980018
stat_clean_confidence     0.952393
spacy_clean_confidence    0.948315
dtype: float64

In [9]:
# Finally, we will compare the vocabulary size and average tokens per review across the different preprocessing methods to understand how each step affects the text data. This will help us see how much information is retained or lost at each stage of cleaning and preprocessing.

def vocab_size(texts):
    vocab = set()
    for text in texts:
        vocab.update(str(text).split())
    return len(vocab)

def avg_tokens(texts):
    return np.mean([len(str(text).split()) for text in texts])

comparison = pd.DataFrame({
    "version": ["raw", "rule_based", "statistical", "spacy"],
    "vocab_size": [
        vocab_size(data["Text"]),
        vocab_size(data["rule_clean"]),
        vocab_size(data["stat_clean"]),
        vocab_size(data["spacy_clean"])
    ],
    "avg_tokens_per_review": [
        avg_tokens(data["Text"]),
        avg_tokens(data["rule_clean"]),
        avg_tokens(data["stat_clean"]),
        avg_tokens(data["spacy_clean"])
    ]
})

comparison


,version,vocab_size,avg_tokens_per_review
0,raw,84548,80.46825
1,rule_based,31402,79.56585
2,statistical,8598,60.85620
3,spacy,20661,35.49315


In [10]:
#cost comparison - Finally, we will compare the computational cost of each preprocessing method by measuring the time taken to process a sample of reviews. This will help us understand the trade-offs between the complexity of the cleaning methods and their impact on the text data, allowing us to make informed decisions about which preprocessing steps to use in our sentiment analysis pipeline.

import time

small = data["Text"].head(1000)

start = time.time()
_ = small.apply(rule_based_clean)
rule_time = time.time() - start

start = time.time()
_ = small.apply(lambda x: statistical_clean(rule_based_clean(x)))
stat_time = time.time() - start

start = time.time()
_ = small.apply(lambda x: spacy_preprocess(rule_based_clean(x)))
spacy_time = time.time() - start

cost_comparison = pd.DataFrame({
    "method": ["rule_based", "statistical", "spacy_deep_preprocessing"],
    "time_seconds_for_1000_reviews": [rule_time, stat_time, spacy_time],
    "scalability": [
        "Fastest; suitable for large datasets",
        "Moderate; requires fitting TF-IDF first",
        "Slowest; better quality but more computationally expensive"
    ]
})

cost_comparison


,method,time_seconds_for_1000_reviews,scalability
0,rule_based,0.117664,Fastest; suitable for large datasets
1,statistical,0.125090,Moderate; requires fitting TF-IDF first
2,spacy_deep_preprocessing,7.769749,Slowest; better quality but more computational...


In [11]:
print("""
Rule-based cleaning removed obvious noise such as HTML tags, URLs, repeated characters,
and unnecessary symbols. It is fast and scalable, but it does not understand meaning.

Statistical filtering reduced vocabulary size by removing very rare and overly common
terms using frequency and TF-IDF thresholds. This improves feature quality, but it may
remove useful domain-specific words if thresholds are too strict.

spaCy preprocessing performed lemmatization and stopword removal while preserving
sentiment-critical tokens such as negations and strong opinion words. This gives cleaner
text while maintaining sentiment meaning, but it is slower than rule-based methods.

Transformer sentiment confidence was used to compare whether the cleaned text still
preserved sentiment clarity. Higher confidence after cleaning suggests that preprocessing
made the sentiment easier for the model to identify.

In terms of computational cost, rule-based cleaning is cheapest, statistical filtering is
moderate, and spaCy/transformer-based preprocessing is the most expensive but often gives
the highest-quality text representation.
""")



Rule-based cleaning removed obvious noise such as HTML tags, URLs, repeated characters,
and unnecessary symbols. It is fast and scalable, but it does not understand meaning.

Statistical filtering reduced vocabulary size by removing very rare and overly common
terms using frequency and TF-IDF thresholds. This improves feature quality, but it may
remove useful domain-specific words if thresholds are too strict.

spaCy preprocessing performed lemmatization and stopword removal while preserving
sentiment-critical tokens such as negations and strong opinion words. This gives cleaner
text while maintaining sentiment meaning, but it is slower than rule-based methods.

Transformer sentiment confidence was used to compare whether the cleaned text still
preserved sentiment clarity. Higher confidence after cleaning suggests that preprocessing
made the sentiment easier for the model to identify.

In terms of computational cost, rule-based cleaning is cheapest, statistical filtering is
moderate, 